<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/7_2_Principal_Component_Analysis_(PCA)_and_SVD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part VII. Dimensionality Reduction and Unsupervised Learning

## Principal Component Analysis (PCA) and SVD

## Clustering: K‑means, Hierarchical, DBSCAN

## Интуиция и геометрия метода главных компонент

### 1. Проблема многомерных данных

Человеческое восприятие ограничено тремя измерениями. Мы легко строим диаграммы рассеяния для двух или трёх признаков, но как быть, если их сотни или тысячи? Такие ситуации типичны: анализ экспрессии генов (тысячи генов), пиксели изображений, текстовые данные в виде мешка слов. Прямая визуализация невозможна, а многие алгоритмы машинного обучения страдают от «проклятия размерности». Хочется сжать данные, сохранив при этом как можно больше полезной информации.

**Метод главных компонент (Principal Component Analysis, PCA)** — один из самых элегантных и широко применяемых способов снижения размерности. Его идея: найти в многомерном пространстве небольшое число направлений, вдоль которых данные меняются сильнее всего, и спроецировать исходные точки на подпространство, натянутое на эти направления. Если данные действительно лежат вблизи некоторой низкоразмерной плоскости, такое проецирование почти не искажает их структуру, зато резко сокращает число признаков.

### 2. Направление максимальной дисперсии

Пусть у нас есть набор из $n$ точек $\mathbf{x}_1,\dots,\mathbf{x}_n \in \mathbb{R}^d$. Для простоты будем считать, что данные центрированы: среднее арифметическое по каждому признаку равно нулю (этого всегда можно добиться вычитанием среднего). Мы ищем прямую, проходящую через начало координат, задаваемую единичным вектором $\mathbf{w}$ ($\|\mathbf{w}\|=1$). Проекция точки $\mathbf{x}_i$ на эту прямую равна скалярному произведению $\mathbf{w}^T\mathbf{x}_i$. Дисперсия проекций

$$
\operatorname{Var}_{\mathbf{w}} = \frac{1}{n}\sum_{i=1}^n (\mathbf{w}^T\mathbf{x}_i)^2
= \mathbf{w}^T\!\left(\frac{1}{n}\sum_{i=1}^n \mathbf{x}_i\mathbf{x}_i^T\right)\!\mathbf{w}
= \mathbf{w}^T \mathbf{\Sigma} \mathbf{w},
$$

где $\mathbf{\Sigma} = \frac{1}{n}\mathbf{X}^T\mathbf{X}$ — выборочная ковариационная матрица размера $d\times d$.

Задача PCA для первой компоненты: найти $\mathbf{w}$, максимизирующий $\mathbf{w}^T \mathbf{\Sigma} \mathbf{w}$ при условии $\|\mathbf{w}\|=1$. Это классическая задача на собственные значения: решением является собственный вектор, отвечающий **наибольшему** собственному числу $\lambda_1$ матрицы $\mathbf{\Sigma}$. Сама дисперсия проекций при этом равна $\lambda_1$.

### 3. Эллипсоид рассеяния и собственные векторы

Ковариационная матрица $\mathbf{\Sigma}$ симметрична и положительно полуопределена. Она задаёт эллипсоид рассеяния данных:

$$
\{ \mathbf{x} \mid \mathbf{x}^T \mathbf{\Sigma}^{-1} \mathbf{x} = \text{const} \}.
$$

Оси этого эллипсоида направлены вдоль собственных векторов $\mathbf{v}_1,\dots,\mathbf{v}_d$ матрицы $\mathbf{\Sigma}$, а их длины пропорциональны $\sqrt{\lambda_i}$. Первая главная компонента — это направление самой длинной оси эллипсоида, т.е. собственный вектор с наибольшим $\lambda$. Вторая компонента — следующая по длине ось, ортогональная первой, и так далее.

Геометрически PCA сводится к повороту исходной системы координат так, чтобы новые оси совпали с осями эллипсоида. Проекция на первые $k$ осей даёт наилучшую $k$-мерную аппроксимацию облака точек в смысле минимизации суммы квадратов расстояний от точек до подпространства.

### 4. Пример: рост и вес

Рассмотрим игрушечный набор данных: рост и вес 200 человек. Пусть рост распределён нормально со средним 170 см и стандартным отклонением 10 см, а вес линейно связан с ростом плюс шум.

Сгенерируем данные, вычислим ковариационную матрицу и её собственные векторы, нанесём на график первую главную компоненту.

```python
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)
n = 200
height = np.random.normal(170, 10, n)
weight = 0.6 * height + np.random.normal(0, 5, n)

# Центрируем данные
X = np.column_stack([height, weight])
X_mean = X.mean(axis=0)
X_centered = X - X_mean

# Ковариационная матрица (смещённая оценка для PCA)
cov_matrix = np.cov(X_centered.T, bias=True)
eigvals, eigvecs = np.linalg.eigh(cov_matrix)

# Первая главная компонента — собственный вектор с максимальным λ
idx = np.argsort(eigvals)[::-1]
eigvals = eigvals[idx]
eigvecs = eigvecs[:, idx]
pc1 = eigvecs[:, 0]

# Линия первой компоненты
line_x = np.linspace(X_centered[:,0].min(), X_centered[:,0].max(), 100)
line_y = (pc1[1] / pc1[0]) * line_x

plt.figure(figsize=(8,6))
plt.scatter(X_centered[:,0], X_centered[:,1], alpha=0.6, label='Данные (центрированные)')
plt.plot(line_x, line_y, 'r-', lw=2, label='Первая главная компонента')
plt.xlabel('Рост (центрированный)')
plt.ylabel('Вес (центрированный)')
plt.title('Рост и вес: первая главная компонента')
plt.legend()
plt.axis('equal')
plt.grid(True)
plt.show()

print("Собственные числа:", eigvals)
print("Первая компонента (вектор):", pc1)
print("Доля объяснённой дисперсии:", eigvals[0] / eigvals.sum())
```

Первая компонента улавливает основную закономерность: с увеличением роста вес в среднем растёт. Угол наклона прямой равен $\arctan(v_2/v_1)$, где $\mathbf{v}=(v_1,v_2)$ — собственный вектор. Вторая компонента (перпендикулярная) соответствует вариации веса при фиксированном росте — индивидуальным отклонениям от средней линии.

### 5. Поворот осей: главные компоненты как новые координаты

Если спроецировать центрированные данные на все главные компоненты, получим новые координаты

$$
\mathbf{Z} = \mathbf{X}_{\text{centered}} \mathbf{V},
$$

где столбцы матрицы $\mathbf{V}$ — собственные векторы $\mathbf{\Sigma}$. В этом базисе ковариационная матрица диагональна: $\frac{1}{n}\mathbf{Z}^T\mathbf{Z} = \operatorname{diag}(\lambda_1,\dots,\lambda_d)$. Это означает, что новые признаки (главные компоненты) некоррелированы. Более того, дисперсия $j$-го признака равна $\lambda_j$. Мы можем отбросить компоненты с малыми $\lambda_j$, теряя минимум информации.

Визуализируем повёрнутые данные для примера «рост–вес»:

```python
# Проекция на обе компоненты
Z = X_centered @ eigvecs

plt.figure(figsize=(8,6))
plt.scatter(Z[:,0], Z[:,1], alpha=0.6)
plt.xlabel('Главная компонента 1')
plt.ylabel('Главная компонента 2')
plt.title('Данные в пространстве главных компонент')
plt.axis('equal')
plt.grid(True)
plt.show()
```

Теперь облако вытянуто вдоль оси абсцисс — именно вдоль неё максимальная дисперсия.

### 6. Минимизация ошибки реконструкции

PCA можно вывести из другой, эквивалентной задачи: найти $k$-мерное линейное подпространство, минимизирующее сумму квадратов расстояний от точек до их проекций на это подпространство (ошибку реконструкции). Если $k=1$, ищется прямая, для которой сумма квадратов расстояний от точек до неё минимальна. Математически это сводится к той же проблеме собственных векторов, но с дополнительным интуитивным смыслом: мы хотим, чтобы сжатое представление $\mathbf{z}_i$ после обратного проецирования $\hat{\mathbf{x}}_i = \mathbf{V}_k \mathbf{z}_i$ (где $\mathbf{V}_k$ — первые $k$ собственных векторов) давало минимальное среднеквадратичное отклонение $\frac{1}{n}\sum_i \|\mathbf{x}_i - \hat{\mathbf{x}}_i\|^2$.

Таким образом, PCA — это не просто «поворот и отрезание», а оптимальный линейный метод сжатия информации с точки зрения сохранения структуры данных.

### 7. Резюме

Метод главных компонент находит новые ортогональные оси, упорядоченные по убыванию дисперсии. Первая ось — направление максимального разброса, вторая — максимального среди оставшихся, перпендикулярного первой, и т.д. Геометрически это сводится к анализу эллипсоида рассеяния, оси которого — собственные векторы ковариационной матрицы. Проецируя данные на первые несколько осей, мы получаем сжатое представление, минимизирующее среднеквадратичную ошибку восстановления.

**Упражнения для читателя:**
1. Почему перед PCA необходимо центрировать данные? Что произойдёт, если этого не сделать?
2. Как изменится первая главная компонента, если рост измерять в метрах, а вес — в фунтах? Как избежать зависимости PCA от масштаба признаков?
3. Докажите, что сумма собственных чисел ковариационной матрицы равна сумме дисперсий исходных признаков. Как это свойство используется для выбора числа компонент?

## Реализация PCA с нуля на NumPy

В предыдущем разделе мы обсудили интуитивный смысл метода главных компонент. Теперь мы перейдём к практической реализации PCA «с нуля», используя только NumPy. Это позволит глубже понять алгоритм, его численные аспекты и подготовит почву для более стабильных методов, основанных на сингулярном разложении.

### 1. Алгоритм по шагам

Пусть дана матрица данных $\mathbf{X} \in \mathbb{R}^{n \times d}$, где $n$ — число наблюдений, $d$ — число признаков. Требуется получить матрицу проекций $\mathbf{Z} \in \mathbb{R}^{n \times k}$ на первые $k$ главных компонент.

**Шаг 1. Центрирование.**  
Вычтем из каждого признака его среднее значение:

$$
\mathbf{X}_{\text{centered}} = \mathbf{X} - \mathbf{1}_n \boldsymbol{\mu}^T, \qquad
\boldsymbol{\mu} = \frac{1}{n} \sum_{i=1}^n \mathbf{x}_i.
$$

Где $\mathbf{1}_n$ — вектор из единиц длины $n$. Центрирование гарантирует, что данные имеют нулевое среднее, что необходимо для корректной работы PCA.

**Шаг 2. Ковариационная матрица.**  
Вычислим выборочную ковариационную матрицу (несмещённая оценка):

$$
\mathbf{C} = \frac{1}{n-1} \mathbf{X}_{\text{centered}}^T \mathbf{X}_{\text{centered}} \;\in\; \mathbb{R}^{d \times d}.
$$

Элемент $C_{ij}$ — ковариация между $i$-м и $j$-м признаками.

**Шаг 3. Собственные значения и векторы.**  
Решаем спектральную задачу:

$$
\mathbf{C} \mathbf{v}_j = \lambda_j \mathbf{v}_j, \quad j=1,\dots,d.
$$

Поскольку $\mathbf{C}$ симметрична и положительно полуопределена, все $\lambda_j \ge 0$, а векторы $\mathbf{v}_j$ можно выбрать ортонормированными.

**Шаг 4. Сортировка.**  
Упорядочим собственные значения по убыванию: $\lambda_1 \ge \lambda_2 \ge \dots \ge \lambda_d \ge 0$ и соответствующие им векторы.

**Шаг 5. Выбор первых $k$ компонент.**  
Образуем матрицу $\mathbf{W}_k \in \mathbb{R}^{d \times k}$, столбцы которой — первые $k$ собственных векторов. Они образуют ортонормированный базис искомого подпространства.

**Шаг 6. Проекция.**  
Новые координаты (главные компоненты) получаются умножением центрированных данных на матрицу весов:

$$
\mathbf{Z} = \mathbf{X}_{\text{centered}} \mathbf{W}_k \;\in\; \mathbb{R}^{n \times k}.
$$

Обратное преобразование: $\hat{\mathbf{X}} = \mathbf{Z} \mathbf{W}_k^T + \boldsymbol{\mu}$, которое восстанавливает данные с минимальной среднеквадратичной ошибкой среди всех линейных $k$-мерных приближений.

### 2. Реализация на Python

Напишем функцию `my_pca`, реализующую описанный алгоритм.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import time

def my_pca(X, k):
    """
    Простой PCA через собственные векторы ковариационной матрицы.
    
    Параметры:
        X : ndarray, shape (n, d)
            Исходные данные (без центрирования).
        k : int
            Число главных компонент (k <= d).
    
    Возвращает:
        X_pca : ndarray, shape (n, k)
            Проекция данных на первые k главных компонент.
        components : ndarray, shape (k, d)
            Главные компоненты (векторы-строки).
        explained_variance : ndarray, shape (k,)
            Дисперсия каждой из k компонент.
        explained_variance_ratio : ndarray, shape (k,)
            Доля объяснённой дисперсии каждой компонентой.
        mean : ndarray, shape (d,)
            Средние значения признаков, использованные для центрирования.
    """
    # Шаг 1: Центрирование
    mean = np.mean(X, axis=0)
    X_centered = X - mean

    # Шаг 2: Ковариационная матрица (несмещённая оценка)
    n = X.shape[0]
    C = (X_centered.T @ X_centered) / (n - 1)

    # Шаг 3: Спектральное разложение
    eig_vals, eig_vecs = np.linalg.eigh(C)  # eigh для симметричных матриц

    # Шаг 4: Сортировка по убыванию собственных значений
    # eigh возвращает в возрастающем порядке, поэтому развернём
    idx = np.argsort(eig_vals)[::-1]
    eig_vals = eig_vals[idx]
    eig_vecs = eig_vecs[:, idx]

    # Шаг 5: Выбор первых k компонент (векторы-столбцы → строки)
    W_k = eig_vecs[:, :k]          # shape (d, k)
    components = W_k.T             # shape (k, d)

    # Шаг 6: Проекция
    X_pca = X_centered @ W_k       # shape (n, k)

    # Вычисление объяснённой дисперсии
    explained_variance = eig_vals[:k]
    explained_variance_ratio = explained_variance / np.sum(eig_vals)

    return X_pca, components, explained_variance, explained_variance_ratio, mean
```

**Замечание о знаках:** Собственные векторы определены с точностью до знака: если $\mathbf{v}$ — собственный вектор, то $-\mathbf{v}$ также собственный вектор. Поэтому знаки главных компонент могут быть противоположными в разных реализациях — это абсолютно нормально и не влияет на геометрический смысл.

### 3. Демонстрация на синтетических данных

Создадим набор данных со скрытой низкоразмерной структурой и коррелированными признаками.

```python
np.random.seed(42)
n, d = 100, 3
# Генерируем скрытые факторы (2 компоненты)
Z_true = np.random.randn(n, 2)
# Случайная матрица нагрузок
W_true = np.random.randn(2, d)
# Данные = факторы * нагрузки + небольшой шум
X = Z_true @ W_true + 0.2 * np.random.randn(n, d)

# Применяем нашу реализацию PCA с k=2
k = 2
X_pca, comps, var, var_ratio, mean = my_pca(X, k)

print("Собственные значения:", var)
print("Доля объяснённой дисперсии:", var_ratio)
print("Кумулятивная доля:", np.sum(var_ratio))
print("Первые две главные компоненты (строки):\n", comps)

# Сравним с sklearn
pca_sk = PCA(n_components=k)
X_pca_sk = pca_sk.fit_transform(X)

print("\nsklearn explained_variance_ratio_:", pca_sk.explained_variance_ratio_)
print("sklearn components:\n", pca_sk.components_)

# Проверим совпадение с точностью до знака
# Абсолютные значения корреляции между компонентами должны быть близки к 1
for i in range(k):
    corr = np.corrcoef(X_pca[:, i], X_pca_sk[:, i])[0, 1]
    print(f"Корреляция между PC{i+1} (моя) и PC{i+1} (sklearn): {corr:.6f}")
```

Вы увидите, что доли объяснённой дисперсии совпадают, а корреляция между соответствующими компонентами равна ±1 с высокой точностью — наша реализация корректна.

### 4. Визуализация и анализ объяснённой дисперсии

Построим график доли объяснённой дисперсии и кумулятивной суммы. Это помогает выбрать $k$.

```python
# Полный PCA (k=d) для анализа дисперсии
_, _, var_all, var_ratio_all, _ = my_pca(X, d)

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.bar(range(1, d+1), var_ratio_all, alpha=0.7, label='Индивидуальная')
plt.step(range(1, d+1), np.cumsum(var_ratio_all), where='mid', label='Кумулятивная')
plt.xlabel('Номер главной компоненты')
plt.ylabel('Доля объяснённой дисперсии')
plt.title('Объяснённая дисперсия')
plt.legend()
plt.grid(True)

plt.subplot(1,2,2)
plt.scatter(X_pca[:, 0], X_pca[:, 1], alpha=0.7)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Данные в пространстве первых двух компонент')
plt.axis('equal')
plt.grid(True)

plt.tight_layout()
plt.show()
```

Из графика видно, что первые две компоненты объясняют почти всю дисперсию — это согласуется с тем, что данные были порождены двумерным латентным процессом.

### 5. Сравнение времени выполнения с sklearn

```python
# Сравнение времени для большого набора данных (n=5000, d=100)
n_big, d_big = 5000, 100
X_big = np.random.randn(n_big, d_big)
k_big = 10

# Наша реализация
start = time.time()
X_pca_my, _, _, _, _ = my_pca(X_big, k_big)
time_my = time.time() - start

# sklearn
pca_sk_big = PCA(n_components=k_big)
start = time.time()
X_pca_sk_big = pca_sk_big.fit_transform(X_big)
time_sk = time.time() - start

print(f"Время выполнения (my_pca): {time_my:.4f} сек")
print(f"Время выполнения (sklearn): {time_sk:.4f} сек")
```

`sklearn` обычно быстрее благодаря использованию SVD и оптимизированных библиотек (LAPACK). Наивный метод через ковариационную матрицу требует $O(d^3)$ на собственные векторы, что может быть дорого при больших $d$.

### 6. Ограничения наивной реализации и переход к SVD

Наш код имеет два серьёзных недостатка.

**Высокая размерность ($d > n$).**  
Если признаков больше, чем наблюдений (типично для геномики, текстов), ковариационная матрица $\mathbf{C}$ имеет размер $d \times d$ и её собственное разложение обходится очень дорого. Вместо этого выгоднее работать с матрицей Грама $\mathbf{G} = \mathbf{X}_{\text{centered}} \mathbf{X}_{\text{centered}}^T$ размера $n \times n$, собственные векторы которой связаны с искомыми компонентами. Ещё лучше — использовать SVD.

**Численная неустойчивость.**  
При малых собственных значениях процедура `eig` может давать отрицательные значения из-за ошибок округления. SVD свободен от этой проблемы и является стандартом де-факто для PCA.

**Связь с SVD** (анонс следующего раздела).  
Пусть $\mathbf{X}_{\text{centered}} = \mathbf{U} \boldsymbol{\Sigma} \mathbf{V}^T$ — усечённое сингулярное разложение, где $\mathbf{U} \in \mathbb{R}^{n \times r}$, $\boldsymbol{\Sigma} \in \mathbb{R}^{r \times r}$, $\mathbf{V} \in \mathbb{R}^{d \times r}$, $r = \min(n,d)$. Тогда:

$$
\mathbf{C} = \frac{1}{n-1} \mathbf{V} \boldsymbol{\Sigma}^2 \mathbf{V}^T,
$$

то есть $\mathbf{V}$ — собственные векторы, а $\lambda_j = \sigma_j^2 / (n-1)$. Проекция: $\mathbf{Z} = \mathbf{U}_k \boldsymbol{\Sigma}_k$. Таким образом, PCA можно выполнить через SVD центрированной матрицы данных, не строя ковариационную матрицу, что значительно стабильнее и эффективнее. Именно так работают `sklearn.decomposition.PCA` и большинство серьёзных библиотек.

### 7. Важное предупреждение: центрирование vs масштабирование

Перед применением PCA данные необходимо **центрировать** (вычесть среднее), иначе первый компонент может уйти в направление среднего, а не вариации. Однако **масштабирование** (деление на стандартное отклонение) не является обязательной частью PCA как такового. Если признаки измеряются в разных единицах (например, возраст в годах и доход в тысячах долларов), то признаки с бо́льшим численным размахом будут доминировать в ковариационной матрице, искажая смысл главных компонент. В таких случаях применяют **стандартизацию** (центрирование + масштабирование до единичной дисперсии), что эквивалентно PCA на корреляционной матрице. Встроенного рецепта нет: выбор зависит от природы данных и целей анализа.

---

### Резюме

Мы построили работающую реализацию PCA «с нуля» на NumPy, следуя классическому алгоритму: центрирование, ковариационная матрица, спектральное разложение, отбор $k$ компонент, проекция. Код протестирован на синтетических данных и сопоставлен с эталонной библиотекой `sklearn`. Проанализирована объяснённая дисперсия, указаны численные и вычислительные ограничения подхода, мотивирующие переход к SVD-реализации, которая будет рассмотрена в следующем разделе.

**Упражнения для читателя:**
1. Почему в формуле ковариационной матрицы мы использовали $n-1$, а не $n$? Изменится ли что-то в PCA, если использовать смещённую оценку?
2. Реализуйте функцию `my_pca_svd(X, k)`, использующую `np.linalg.svd` для центрированной матрицы, и сравните точность с `my_pca` на данных, где $d \approx n$.
3. Исследуйте, как меняются главные компоненты, если перед PCA признаки стандартизировать. В каком случае это необходимо?